# Unified COCO-style evaluation — VisDrone, maxDet=500

This notebook evaluates **Faster R-CNN, YOLO, and RT-DETR** on exactly the same processed VisDrone validation set. Add the processed VisDrone dataset and the three trained model outputs as Kaggle Inputs, enable a GPU, then choose **Run All**.

Protocol: IoU thresholds `0.50:0.05:0.95`, 101 recall points, COCO area ranges (`small < 32²`, `medium = 32²–96²`, `large >= 96²`), and `maxDets=[1,10,100,500]`. AP/AP50/AP75 and every size-specific AP/AR use **maxDet=500**. This is a COCO-style extension for crowded VisDrone images, not the official COCO maxDet=100 table and not the VisDrone official ignored-region evaluator. Values in the final table are percentages.

## 1. Configuration

Normally the paths can remain `None`; the notebook searches Kaggle Inputs. If auto-discovery reports multiple candidates, copy the desired path from its error message into the corresponding variable. Weight files must use the same 10 VisDrone classes and class names as the processed ground truth.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Nhattk19/Object_dectection.git"
REPO_BRANCH = "cnn-faster-rcnn-pipeline"
PROJECT_ROOT_OVERRIDE = None       # e.g. /kaggle/input/object-detection-source/Object_dectection
PROCESSED_DATA_ROOT = None         # folder containing annotations/instances_val.json and images/

# Fill a path only when auto-discovery is ambiguous. Always select best.pth, not last.pth.
FASTER_RCNN_CHECKPOINTS = {
    "F0": None,  # e.g. /kaggle/input/cnn-checkpoints/f0/best.pth
    "F1": None,  # e.g. /kaggle/input/cnn-checkpoints/f1/best.pth
    "F2": None,  # e.g. /kaggle/input/cnn-checkpoints/f2/best.pth
    "F3": None,  # e.g. /kaggle/input/cnn-checkpoints/f3/best.pth
    "F4": None,  # e.g. /kaggle/input/cnn-checkpoints/f4/best.pth
    "F5": None,  # e.g. /kaggle/input/cnn-checkpoints/f5/best.pth
}
YOLO_CHECKPOINT = None             # e.g. /kaggle/input/yolo-output/runs/detect/train/weights/best.pt
RTDETR_CHECKPOINT = None           # e.g. /kaggle/input/rtdetr-output/runs/detect/train/weights/best.pt

YOLO_IMGSZ = 960
RTDETR_IMGSZ = 960
ULTRALYTICS_BATCH = 4              # reduce to 1 or 2 if GPU memory is insufficient
WORKERS = 2
SCORE_THRESHOLD = 0.001            # low threshold is required for fair AP/AR integration
NMS_IOU = 0.70
MAX_DETECTIONS = 500
REUSE_PREDICTIONS = True
RUN_F5_TEST_DEV = True             # test only the frozen F5 selected on validation
TEST_DEV_ANNOTATION_FILE = None    # normally DATA_ROOT/annotations/instances_test-dev.json
OUTPUT_ROOT = Path("/kaggle/working/coco_eval_all_maxdet500")

EVALUATE_YOLO_AND_RTDETR = True    # set False when this run is only the CNN ablation
MODEL_SPECS = [
    {"name": f"Faster R-CNN {experiment}", "kind": "faster_rcnn",
     "experiment": experiment, "path": checkpoint}
    for experiment, checkpoint in FASTER_RCNN_CHECKPOINTS.items()
]
if EVALUATE_YOLO_AND_RTDETR:
    MODEL_SPECS += [
        {"name": "YOLO", "kind": "yolo", "path": YOLO_CHECKPOINT, "imgsz": YOLO_IMGSZ},
        {"name": "RT-DETR", "kind": "rtdetr", "path": RTDETR_CHECKPOINT, "imgsz": RTDETR_IMGSZ},
    ]
assert MAX_DETECTIONS == 500, "This report protocol requires maxDet=500"

## 2. Set up the repository and dependencies

In [ ]:
import importlib.util
import subprocess
import sys

def find_project_root():
    if PROJECT_ROOT_OVERRIDE:
        root = Path(PROJECT_ROOT_OVERRIDE)
        if not (root / "src/coco_evaluation.py").is_file():
            raise FileNotFoundError(f"Invalid PROJECT_ROOT_OVERRIDE: {root}")
        return root.resolve()
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        matches = list(base.glob("**/src/coco_evaluation.py"))
        if matches:
            return matches[0].parents[1].resolve()
    target = Path("/kaggle/working/Object_dectection")
    try:
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(target)])
    except subprocess.CalledProcessError as error:
        raise RuntimeError("Repository clone failed. Turn Internet on, or Add Input containing this repository and set PROJECT_ROOT_OVERRIDE.") from error
    return target.resolve()

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

missing = []
if importlib.util.find_spec("pycocotools") is None:
    missing.append("pycocotools>=2.0.7")
if any(spec["kind"] in {"yolo", "rtdetr"} for spec in MODEL_SPECS) and importlib.util.find_spec("ultralytics") is None:
    missing.append("ultralytics>=8.3")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import torch
from coco_evaluation import (
    COCO_MAX_DETS, COCO_METRIC_NAMES, evaluate_coco_predictions,
    load_coco_predictions, metrics_as_percent, run_faster_rcnn_predictions,
    run_ultralytics_predictions, save_coco_predictions,
)
assert tuple(COCO_MAX_DETS) == (1, 10, 100, 500)
assert torch.cuda.is_available(), "Enable GPU Accelerator in Kaggle Settings"
print({"project": str(PROJECT_ROOT), "gpu": torch.cuda.get_device_name(0), "maxDets": COCO_MAX_DETS})

## 3. Locate and validate the common validation set

The selected COCO file must describe the processed VisDrone validation split. The image IDs, category IDs, bounding boxes, and area groups from this one file are used for every model.

In [ ]:
import json

def inspect_processed_root(root):
    root = Path(root)
    annotation = root / "annotations/instances_val.json"
    image_root = root / "images"
    if not annotation.is_file() or not image_root.is_dir():
        return None
    payload = json.loads(annotation.read_text())
    if not payload.get("images") or len(payload.get("categories", [])) != 10:
        return None
    sample = image_root / payload["images"][0]["file_name"]
    if not sample.is_file():
        return None
    return root.resolve(), annotation.resolve(), image_root.resolve(), payload

if PROCESSED_DATA_ROOT:
    found = inspect_processed_root(PROCESSED_DATA_ROOT)
    if found is None:
        raise FileNotFoundError(f"Invalid PROCESSED_DATA_ROOT: {PROCESSED_DATA_ROOT}")
else:
    candidates = []
    for annotation in Path("/kaggle/input").glob("**/annotations/instances_val.json"):
        inspected = inspect_processed_root(annotation.parents[1])
        if inspected is not None:
            candidates.append(inspected)
    unique = {str(item[0]): item for item in candidates}
    if len(unique) != 1:
        raise RuntimeError("Expected exactly one processed VisDrone input. Candidates: " + repr(sorted(unique)))
    found = next(iter(unique.values()))

DATA_ROOT, ANNOTATION_FILE, IMAGE_ROOT, ground_truth = found
image_ids = {int(item["id"]) for item in ground_truth["images"]}
category_ids = {int(item["id"]) for item in ground_truth["categories"]}
assert category_ids == set(range(1, 11)), f"Unexpected category IDs: {category_ids}"
print({
    "data_root": str(DATA_ROOT), "annotation": str(ANNOTATION_FILE),
    "images": len(image_ids), "annotations": len(ground_truth["annotations"]),
    "categories": [item["name"] for item in ground_truth["categories"]],
})

## 4. Resolve the three model artifacts

Auto-discovery deliberately refuses ambiguous matches. This prevents accidentally evaluating `last.pt`, an F0 checkpoint, or another model's `best.pt`.

In [ ]:
def discover_checkpoint(spec):
    kind = spec["kind"]
    files = [p for p in Path("/kaggle/input").glob("**/*") if p.is_file()]
    if kind == "faster_rcnn":
        experiment = spec["experiment"].lower()
        candidates = [p for p in files if p.name == "best.pth" and experiment in [part.lower() for part in p.parts]]
    elif kind == "yolo":
        candidates = [p for p in files if p.name == "best.pt" and "yolo" in str(p).lower() and "rtdetr" not in str(p).lower()]
    elif kind == "rtdetr":
        candidates = [p for p in files if p.name == "best.pt" and any(token in str(p).lower() for token in ("rtdetr", "rt-detr", "rt_detr"))]
    elif kind == "coco_json":
        candidates = [p for p in files if p.suffix == ".json" and "prediction" in p.name.lower()]
    else:
        raise ValueError(f"Unknown model kind: {kind}")
    if len(candidates) != 1:
        label = spec.get("experiment", kind)
        raise RuntimeError(f"{label}: expected one artifact, found {len(candidates)}. Set its path explicitly. Candidates: {candidates}")
    return candidates[0].resolve()

for spec in MODEL_SPECS:
    spec["path"] = Path(spec["path"]).resolve() if spec.get("path") else discover_checkpoint(spec)
    if not spec["path"].is_file():
        raise FileNotFoundError(spec["path"])
    print(f"{spec['name']:20s} | {spec['kind']:12s} | {spec['path']}")

## 5. Inference and unified COCO evaluation

Each adapter exports the same COCO result schema: `image_id`, `category_id`, `bbox=[x,y,w,h]`, and `score`. The shared evaluator then computes all metrics. Cached prediction JSON files make it possible to rerun evaluation without repeating inference.

In [ ]:
import gc
import re
import pandas as pd

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "predictions").mkdir(exist_ok=True)
(OUTPUT_ROOT / "runtime").mkdir(exist_ok=True)

def slugify(value):
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")

rows, runtime_records = [], []
for spec in MODEL_SPECS:
    slug = slugify(spec["name"])
    prediction_file = OUTPUT_ROOT / "predictions" / f"{slug}.json"
    runtime_file = OUTPUT_ROOT / "runtime" / f"{slug}.json"
    print(f"\n=== {spec['name']} ===")
    if REUSE_PREDICTIONS and prediction_file.is_file():
        predictions = load_coco_predictions(prediction_file)
        runtime = json.loads(runtime_file.read_text()) if runtime_file.is_file() else {"adapter": "cached"}
        print(f"Reusing {len(predictions):,} predictions")
    elif spec["kind"] == "coco_json":
        predictions = load_coco_predictions(spec["path"])
        runtime = {"adapter": "coco_json", "source": str(spec["path"]), "predictions": len(predictions)}
    elif spec["kind"] == "faster_rcnn":
        predictions, runtime = run_faster_rcnn_predictions(
            spec["path"], ANNOTATION_FILE, IMAGE_ROOT, workers=WORKERS,
            score_threshold=SCORE_THRESHOLD, max_detections=MAX_DETECTIONS,
        )
    else:
        predictions, runtime = run_ultralytics_predictions(
            spec["path"], spec["kind"], ANNOTATION_FILE, IMAGE_ROOT,
            imgsz=spec["imgsz"], batch_size=ULTRALYTICS_BATCH,
            score_threshold=SCORE_THRESHOLD, nms_iou=NMS_IOU,
            max_detections=MAX_DETECTIONS,
        )
    save_coco_predictions(prediction_file, predictions)
    runtime_file.write_text(json.dumps(runtime, indent=2))
    predicted_image_ids = {int(item["image_id"]) for item in predictions}
    unknown = predicted_image_ids - image_ids
    assert not unknown, f"Predictions contain unknown image IDs: {sorted(unknown)[:10]}"
    metrics = evaluate_coco_predictions(
        ANNOTATION_FILE, predictions, image_ids=sorted(image_ids),
        max_dets=(1, 10, 100, 500), ap_max_dets=500,
    )
    row = {"Model": spec["name"], **metrics_as_percent(metrics)}
    rows.append(row)
    runtime_records.append({"Model": spec["name"], **runtime})
    display(pd.DataFrame([row]).round(3))
    gc.collect()
    torch.cuda.empty_cache()

results = pd.DataFrame(rows, columns=["Model", *COCO_METRIC_NAMES])
runtime_table = pd.DataFrame(runtime_records)

## 6. Results and Kaggle Output

In [ ]:
import matplotlib.pyplot as plt

display(results.round(3).style.highlight_max(axis=0, subset=list(COCO_METRIC_NAMES), color="#c6efce"))
display(runtime_table)

results.to_csv(OUTPUT_ROOT / "coco_metrics_all.csv", index=False)
(OUTPUT_ROOT / "coco_metrics_all.json").write_text(results.to_json(orient="records", indent=2))
runtime_table.to_csv(OUTPUT_ROOT / "runtime_all.csv", index=False)
protocol = {
    "name": "COCO-style bbox evaluation extended to maxDet=500",
    "annotation_file": str(ANNOTATION_FILE),
    "num_images": len(image_ids),
    "iou_thresholds": [round(x / 100, 2) for x in range(50, 96, 5)],
    "recall_points": 101, "maxDets": [1, 10, 100, 500], "ap_maxDet": 500,
    "area_ranges_px2": {"small": [0, 32**2], "medium": [32**2, 96**2], "large": [96**2, "inf"]},
    "score_threshold": SCORE_THRESHOLD,
    "note": "COCO-style extension; not official COCO maxDet=100 or VisDrone official evaluation",
}
(OUTPUT_ROOT / "evaluation_protocol.json").write_text(json.dumps(protocol, indent=2))

ax = results.set_index("Model")[["AP (50:95)", "AP50", "AP75", "AR@500"]].plot.bar(figsize=(10, 5), rot=0)
ax.set_ylabel("Score (%)")
ax.set_title("Unified COCO-style evaluation (maxDet=500)")
ax.grid(axis="y", alpha=.25)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "model_comparison.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved Kaggle Output to:", OUTPUT_ROOT)

## 7. Frozen F5 evaluation on test-dev

F5 has already been selected using validation. This cell evaluates that frozen `best.pth` once on test-dev; it must not be used to choose another F configuration or epoch.

In [ ]:
if RUN_F5_TEST_DEV:
    f5_matches = [spec for spec in MODEL_SPECS if spec.get("experiment") == "F5"]
    if len(f5_matches) != 1:
        raise RuntimeError(f"Expected one resolved F5 specification, found {f5_matches}")
    f5_spec = f5_matches[0]
    test_annotation = (
        Path(TEST_DEV_ANNOTATION_FILE).resolve()
        if TEST_DEV_ANNOTATION_FILE
        else DATA_ROOT / "annotations/instances_test-dev.json"
    )
    if not test_annotation.is_file():
        alternatives = list(Path("/kaggle/input").glob("**/annotations/instances_test-dev.json"))
        raise FileNotFoundError(f"Missing {test_annotation}. Available candidates: {alternatives}")

    test_ground_truth = json.loads(test_annotation.read_text())
    test_image_ids = {int(item["id"]) for item in test_ground_truth["images"]}
    test_category_ids = {int(item["id"]) for item in test_ground_truth["categories"]}
    assert test_category_ids == set(range(1, 11)), f"Unexpected test-dev category IDs: {test_category_ids}"
    assert test_image_ids and test_image_ids.isdisjoint(image_ids), "Validation and test-dev image IDs must be disjoint"
    for info in test_ground_truth["images"][:10]:
        path = IMAGE_ROOT / info["file_name"]
        if not path.is_file():
            raise FileNotFoundError(f"Missing test-dev image: {path}")

    test_output = OUTPUT_ROOT / "test_dev" / "f5"
    test_output.mkdir(parents=True, exist_ok=True)
    test_prediction_file = test_output / "predictions.json"
    test_runtime_file = test_output / "runtime.json"
    reuse_test = False
    if REUSE_PREDICTIONS and test_prediction_file.is_file() and test_runtime_file.is_file():
        cached_runtime = json.loads(test_runtime_file.read_text())
        reuse_test = cached_runtime.get("checkpoint") == str(f5_spec["path"])
    if reuse_test:
        test_predictions = load_coco_predictions(test_prediction_file)
        test_runtime = cached_runtime
        print(f"Reusing {len(test_predictions):,} cached F5 test-dev predictions")
    else:
        test_predictions, test_runtime = run_faster_rcnn_predictions(
            f5_spec["path"], test_annotation, IMAGE_ROOT, workers=WORKERS,
            score_threshold=SCORE_THRESHOLD, max_detections=MAX_DETECTIONS,
        )
        save_coco_predictions(test_prediction_file, test_predictions)
        test_runtime_file.write_text(json.dumps(test_runtime, indent=2))

    unknown_test_ids = {int(item["image_id"]) for item in test_predictions} - test_image_ids
    assert not unknown_test_ids, f"Predictions contain unknown test-dev IDs: {sorted(unknown_test_ids)[:10]}"
    test_metrics = evaluate_coco_predictions(
        test_annotation, test_predictions, image_ids=sorted(test_image_ids),
        max_dets=(1, 10, 100, 500), ap_max_dets=500,
    )
    f5_test_results = pd.DataFrame(
        [{"Model": "Faster R-CNN F5", "Split": "test-dev", **metrics_as_percent(test_metrics)}],
        columns=["Model", "Split", *COCO_METRIC_NAMES],
    )
    display(f5_test_results.round(3))
    f5_test_results.to_csv(test_output / "coco_metrics_f5_test_dev.csv", index=False)
    (test_output / "coco_metrics_f5_test_dev.json").write_text(f5_test_results.to_json(orient="records", indent=2))
    test_protocol = {**protocol, "split": "test-dev", "annotation_file": str(test_annotation), "num_images": len(test_image_ids), "checkpoint": str(f5_spec["path"])}
    (test_output / "evaluation_protocol.json").write_text(json.dumps(test_protocol, indent=2))
    print({"test_images": len(test_image_ids), "test_annotations": len(test_ground_truth["annotations"]), "output": str(test_output)})
else:
    print("F5 test-dev evaluation disabled")

After the run finishes, choose **Save Version → Save & Run All (Commit)** so `/kaggle/working/coco_eval_all_maxdet500` becomes a persistent notebook Output. Validation ablations remain in `coco_metrics_all.csv`; the frozen F5 test result is saved separately under `test_dev/f5/`. Keep both protocol files and prediction JSON files for reproducibility.